# Sequence Predictions
This notebook converts each review description into a sequence of integers using Tokenizer

These sequences are then padded so they are all equal length

A Recurrent Neural Network is trained on this data to predict the wine variety

## Filter Data

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('Wine Reviews Dataset/winemag-data-130k-v2.csv')

varieties = [
    'Pinot Noir',
    'Chardonnay',
    'Cabernet Sauvignon',
    'Red Blend',
    'Bordeaux-style Red Blend',
    'Riesling',
    'Sauvignon Blanc',
    'Syrah',
    'Rosé',
    'Merlot'
]

# Keep only the rows with the varieties listed above
df = df[df['variety'].isin(varieties)]

# Replace the wine names with [WINE] so the model doesn't just learn the names of the wines
df['description'] = df['description'].replace(varieties, '[WINE]')

# Discard columns we won't be using
df = df.drop(columns=['Unnamed: 0', 'country', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'winery'])

# Encode labels
le = LabelEncoder()
df['variety'] = le.fit_transform(df['variety'])

df[0:10]

,description,variety
3,"Pineapple rind, lemon pith and orange blossom ...",6
4,"Much like the regular bottling from 2012, this...",4
10,"Soft, supple plum envelopes an oaky structure ...",1
12,"Slightly reduced, this wine offers a chalky, t...",1
14,Building on 150 years and six generations of w...,2
15,Zesty orange peels and apple notes abound in t...,6
20,Ripe aromas of dark berries mingle with ample ...,5
21,"A sleek mix of tart berry, stem and herb, alon...",4
23,This wine from the Geneseo district offers aro...,3
25,Oak and earth intermingle around robust aromas...,4


## Tokenize features and labels

In [3]:
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split

y = df['variety']

# Feature tokenizer
feature_tokenizer = Tokenizer()
feature_tokenizer.fit_on_texts(df['description'])
X = feature_tokenizer.texts_to_sequences(df['description'])

# Add padding to ensure all sequences have the same length
X = pad_sequences(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

## Recurrent Neural Network

In [5]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Embedding, LSTM, Masking

# Create the neural network model
model = Sequential()
model.add(Embedding(len(feature_tokenizer.word_index) + 1, 128, input_length=X_train.shape[1], mask_zero=True))
model.add(Masking(mask_value=0.0))
model.add(LSTM(32, return_sequences=True))
model.add(LSTM(16))
# model.add(Dense(32, activation='relu'))
# model.add(Dense(16, activation='relu'))
# model.add(Dropout(0.5))
model.add(Dense(len(varieties), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=16, validation_data=(X_test, y_test))

Epoch 1/10


2023-11-26 17:26:39.521165: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2023-11-26 17:26:39.732026: W tensorflow/core/common_runtime/type_inference.cc:339] Type inference failed. This indicates an invalid graph that escaped type checking. Error message: INVALID_ARGUMENT: expected compatible input types, but input 1:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_INT32
    }
  }
}
 is neither a subtype nor a supertype of the combined inputs preceding it:
type_id: TFT_OPTIONAL
args {
  type_id: TFT_PRODUCT
  args {
    type_id: TFT_TENSOR
    args {
      type_id: TFT_FLOAT
    }
  }
}

	while inferring type of node 'cond_40/output/_23'
2023-11-26 17:26:40.183416: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x7f2970527a80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-11-26 17:26:40.183440: I tensorflow/compiler/

 106/3567 [..............................] - ETA: 7:35 - loss: 2.1981 - accuracy: 0.1916

KeyboardInterrupt: 

## Transformer

In [2]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import logging

logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# Rename columns to fit the simpletransformers API
df = df.rename(columns={'description': 'text', 'variety': 'labels'})
# Split the dataframe into train and test sets
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=12)

# Optional model configuration
model_args = ClassificationArgs(
    num_train_epochs=5,
    use_multiprocessing=False,
    use_multiprocessing_for_evaluation=False,
)

# Create a ClassificationModel
model = ClassificationModel(
    "bert", "bert-base-cased", num_labels=10, args=model_args
)

# Train the model
model.train_model(train_df)

# Evaluate the model
result, model_outputs, wrong_predictions = model.eval_model(eval_df)

# Make predictions with the model
predictions, raw_outputs = model.predict(eval_df['text'][0:10].tolist())

print(f"Accuracy? I think?: {(len(model_outputs) - len(wrong_predictions)) / len(model_outputs)}")

2023-11-26 20:43:26.685136: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-11-26 20:43:27.346103: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.
INFO:simpletransformers.classification.classification_utils: Saving features into cached file cache_dir/cached_train_bert_128_10_2


Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

Running Epoch 0 of 5:   0%|          | 0/7133 [00:00<?, ?it/s]

Running Epoch 1 of 5:   0%|          | 0/7133 [00:00<?, ?it/s]

Running Epoch 2 of 5:   0%|          | 0/7133 [00:00<?, ?it/s]

...

INFO:simpletransformers.classification.classification_model: Training of bert model complete. Saved to outputs/.
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.
INFO:simpletransformers.classification.classification_utils: Saving features into cached file cache_dir/cached_dev_bert_128_10_2


Running Evaluation:   0%|          | 0/1784 [00:00<?, ?it/s]

INFO:simpletransformers.classification.classification_model:{'mcc': 0.8175391709364216, 'eval_loss': 0.910357636136458}
INFO:simpletransformers.classification.classification_utils: Converting to features started. Cache is not used.


  0%|          | 0/2 [00:00<?, ?it/s]

[0 1 6 2 1 8 0 8 4 8]
53372     0
54862     1
16522     6
109669    2
109560    1
49562     8
98704     0
116629    8
87439     0
125774    6
Name: labels, dtype: int64
Accuracy? I think?: 0.8401682439537329
